# Colab Fine-Tuning: ESM-2 8M Mean-Pooling Classifier

This notebook fine-tunes `facebook/esm2_t6_8M_UR50D` end to end for soluble-versus-membrane classification.

Before running, choose **Runtime → Change runtime type → GPU**. The authoritative training implementation is `scripts/train_finetune.py`; this notebook handles the Colab environment, data upload, execution, and result download.

## 1. Clone the GitHub repository

This cell clones the public GitHub repository. For a private repository, Colab will require an authenticated clone method.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/yangmei25/esm2-protein-localization.git"
REPO_DIR = Path("/content/esm2-protein-localization")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print("Repository already exists:", REPO_DIR)
os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())

## 2. Install fine-tuning dependencies

Colab supplies PyTorch and the CUDA runtime. This cell installs the remaining project libraries without replacing Colab's GPU-enabled PyTorch build.

In [ ]:
import sys
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.40,<5", "pandas>=2,<3", "scikit-learn>=1.3,<2", "tqdm>=4.65",
], check=True)
print("Dependencies installed. If imports fail, restart the runtime and rerun from cell 1.")

## 3. Verify the GPU

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Select a GPU runtime before training.")
print("GPU:", torch.cuda.get_device_name(0))
properties = torch.cuda.get_device_properties(0)
print(f"GPU memory: {properties.total_memory / 1024**3:.1f} GB")

## 4. Mount Google Drive

Training outputs will be written directly to `MyDrive/esm2-protein-localization/finetune-results/` so checkpoints and metrics survive a Colab runtime disconnect.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/esm2-protein-localization")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Google Drive output root:", DRIVE_ROOT)

## 5. Upload the processed dataset

Project data is intentionally excluded from GitHub. Upload your local file:

`data/processed/deeploc_binary.csv`

The upload is about 3 MB. It contains the fixed train, validation, and test assignments used by the frozen ESM-2 experiment.

In [ ]:
from google.colab import files
import shutil

DATA_PATH = REPO_DIR / "data/processed/deeploc_binary.csv"
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
if not DATA_PATH.exists():
    uploaded = files.upload()
    if "deeploc_binary.csv" not in uploaded:
        raise ValueError("Please upload the file named deeploc_binary.csv")
    shutil.move("deeploc_binary.csv", DATA_PATH)
print("Dataset ready:", DATA_PATH, f"({DATA_PATH.stat().st_size / 1024**2:.2f} MB)")

In [ ]:
import pandas as pd
data = pd.read_csv(DATA_PATH)
display(data.groupby(["split", "label"]).size().unstack(fill_value=0).rename(columns={0: "soluble", 1: "membrane"}))
print("Proteins:", len(data))
print("Maximum length:", data["original_length"].max())
assert len(data) == 7_890
assert data["original_length"].max() <= 1_022

## 6. Configure the run

The defaults target a typical Colab GPU. If CUDA runs out of memory, reduce `BATCH_SIZE` to 2 or 1 and increase `GRADIENT_ACCUMULATION` so the effective batch size remains similar.

In [ ]:
OUTPUT_DIR = DRIVE_ROOT / "finetune-results/esm2_t6_8M_mean"
EPOCHS = 5
BATCH_SIZE = 4
EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION = 4
LEARNING_RATE = 2e-5
EARLY_STOPPING_PATIENCE = 2

print("Effective training batch size:", BATCH_SIZE * GRADIENT_ACCUMULATION)
print("Output directory:", OUTPUT_DIR)

## 7. Fine-tune ESM-2

This cell downloads the pretrained 8M checkpoint on first use, updates the full encoder and classification head, evaluates validation data after every epoch, and saves the checkpoint with the best validation F1. It intentionally does not evaluate test data.

In [ ]:
command = [
    sys.executable, "-u", "scripts/train_finetune.py",
    "--data", str(DATA_PATH),
    "--output-dir", str(OUTPUT_DIR),
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--eval-batch-size", str(EVAL_BATCH_SIZE),
    "--gradient-accumulation-steps", str(GRADIENT_ACCUMULATION),
    "--learning-rate", str(LEARNING_RATE),
    "--early-stopping-patience", str(EARLY_STOPPING_PATIENCE),
    "--device", "cuda",
    "--mixed-precision", "auto",
]
print("Running:", " ".join(command), flush=True)
subprocess.run(command, cwd=REPO_DIR, check=True)

## 8. Inspect validation history

In [ ]:
import json
import matplotlib.pyplot as plt

history = pd.read_csv(OUTPUT_DIR / "history.csv")
best_metrics = json.loads((OUTPUT_DIR / "best_validation_metrics.json").read_text())
display(history)
print("Best validation metrics:")
print(json.dumps(best_metrics, indent=2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].plot(history["epoch"], history["train_loss"], "o-", label="Train")
axes[0].plot(history["epoch"], history["validation_loss"], "o-", label="Validation")
axes[0].set(title="Loss", xlabel="Epoch", ylabel="Cross-entropy loss")
axes[0].legend(frameon=False)
for metric in ["validation_f1", "validation_accuracy", "validation_roc_auc"]:
    axes[1].plot(history["epoch"], history[metric], "o-", label=metric.replace("validation_", ""))
axes[1].set(title="Validation metrics", xlabel="Epoch", ylabel="Score", ylim=(0, 1.02))
axes[1].legend(frameon=False)
for axis in axes:
    axis.spines[["top", "right"]].set_visible(False)
plt.show()

## 9. Optional exploratory test evaluation

The official test split has already been inspected during earlier representation comparisons. Leave this disabled until you have reviewed and accepted the validation-selected checkpoint. Enabling it loads the best checkpoint without retraining.

In [ ]:
RUN_EXPLORATORY_TEST = False

if RUN_EXPLORATORY_TEST:
    subprocess.run([
        sys.executable, "-u", "scripts/train_finetune.py",
        "--data", str(DATA_PATH),
        "--output-dir", str(OUTPUT_DIR),
        "--device", "cuda",
        "--mixed-precision", "auto",
        "--test-only",
    ], cwd=REPO_DIR, check=True)
    test_metrics = json.loads((OUTPUT_DIR / "test_metrics.json").read_text())
    print(json.dumps(test_metrics, indent=2))
else:
    print("Test evaluation skipped. Set RUN_EXPLORATORY_TEST = True when intentional.")

## 10. Download the checkpoint and results

The individual result files are already safe in Google Drive. This optional step also creates a ZIP in the same Drive folder and downloads a local copy.

In [ ]:
archive_base = DRIVE_ROOT / "esm2_t6_8M_mean_results"
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=OUTPUT_DIR)
print("Created:", archive_path)
files.download(archive_path)

## Handoff

The results remain in `MyDrive/esm2-protein-localization/finetune-results/esm2_t6_8M_mean/`. After the run, copy them into `results/finetune/esm2_t6_8M_mean/` locally if you want them in the project workspace. Review validation performance before making further hyperparameter decisions. Do not claim improvement until the fine-tuned model is compared with frozen ESM-2 on the same proteins, split, threshold, and metrics.